In [1]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


In [2]:
# Build a .py script that takes a snapshot date, trains a model and outputs artefact into storage.

## set up pyspark session

In [3]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-arm64"
os.environ["PATH"] += os.pathsep + os.path.join(os.environ["JAVA_HOME"], "bin")

# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/25 16:27:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## set up config

In [4]:
# set up config
# model_train_date_str = "2023-01-01"
model_train_date_str = "2024-09-01"
train_test_period_months = 12
oot_period_months = 2
train_test_ratio = 0.8

config = {}
config["model_train_date_str"] = model_train_date_str
config["train_test_period_months"] = train_test_period_months
config["oot_period_months"] =  oot_period_months
config["model_train_date"] =  datetime.strptime(model_train_date_str, "%Y-%m-%d")
config["oot_end_date"] =  config['model_train_date'] - timedelta(days = 1)
config["oot_start_date"] =  config['model_train_date'] - relativedelta(months = oot_period_months)
config["train_test_end_date"] =  config["oot_start_date"] - timedelta(days = 1)
config["train_test_start_date"] =  config["oot_start_date"] - relativedelta(months = train_test_period_months)
config["train_test_ratio"] = train_test_ratio 


pprint.pprint(config)

{'model_train_date': datetime.datetime(2024, 9, 1, 0, 0),
 'model_train_date_str': '2024-09-01',
 'oot_end_date': datetime.datetime(2024, 8, 31, 0, 0),
 'oot_period_months': 2,
 'oot_start_date': datetime.datetime(2024, 7, 1, 0, 0),
 'train_test_end_date': datetime.datetime(2024, 6, 30, 0, 0),
 'train_test_period_months': 12,
 'train_test_ratio': 0.8,
 'train_test_start_date': datetime.datetime(2023, 7, 1, 0, 0)}


## get label store

In [5]:
# connect to label store
folder_path = "datamart/gold/label_store/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
label_store_sdf = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",label_store_sdf.count())

label_store_sdf.show()

row_count: 8974


+-----------+-----+----------+-------------+
|Customer_ID|label| label_def|snapshot_date|
+-----------+-----+----------+-------------+
| CUS_0x1037|    0|30dpd_6mob|   2023-07-01|
| CUS_0x1069|    0|30dpd_6mob|   2023-07-01|
| CUS_0x114a|    0|30dpd_6mob|   2023-07-01|
| CUS_0x1184|    0|30dpd_6mob|   2023-07-01|
| CUS_0x1297|    1|30dpd_6mob|   2023-07-01|
| CUS_0x12fb|    0|30dpd_6mob|   2023-07-01|
| CUS_0x1325|    0|30dpd_6mob|   2023-07-01|
| CUS_0x1341|    0|30dpd_6mob|   2023-07-01|
| CUS_0x1375|    1|30dpd_6mob|   2023-07-01|
| CUS_0x13a8|    0|30dpd_6mob|   2023-07-01|
| CUS_0x13ef|    0|30dpd_6mob|   2023-07-01|
| CUS_0x1440|    0|30dpd_6mob|   2023-07-01|
| CUS_0x1443|    0|30dpd_6mob|   2023-07-01|
| CUS_0x145a|    0|30dpd_6mob|   2023-07-01|
| CUS_0x1492|    0|30dpd_6mob|   2023-07-01|
| CUS_0x153d|    0|30dpd_6mob|   2023-07-01|
| CUS_0x1567|    0|30dpd_6mob|   2023-07-01|
| CUS_0x15ad|    1|30dpd_6mob|   2023-07-01|
| CUS_0x1630|    1|30dpd_6mob|   2023-07-01|
| CUS_0x16

In [6]:
# extract label store
# labels_sdf = label_store_sdf.filter((col("snapshot_date") >= config["train_test_start_date"]) & (col("snapshot_date") <= config["oot_end_date"]))
# print("extracted labels_sdf", labels_sdf.count(), config["train_test_start_date"], config["oot_end_date"])

labels_sdf = label_store_sdf
print("extracted labels_sdf", labels_sdf.count())

[Stage 5:=============================>                             (4 + 4) / 8]

extracted labels_sdf 8974


## get features

In [7]:
folder_path = "datamart/gold/feature_store/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
features_store_sdf = spark.read.option("header", "true").parquet(*files_list)
features_store_sdf = features_store_sdf.drop(
    "attributes_snapshot_date", "financials_snapshot_date", "clickstream_snapshot_date"
)
features_store_sdf = features_store_sdf.withColumnRenamed(
    "snapshot_date","feature_snapshot_date"
)

print("row_count:",features_store_sdf.count())

features_store_sdf.show()


row_count: 7848
+-----------+---------------------+---+-------------+-------------+---------------------+-----------------+---------------+-------------+-----------+--------------------+-------------------+----------------------+--------------------+--------------------+----------+----------------+------------------------+--------------------+---------------------+-------------------+-----------------------+--------------------+---------------+---------------------+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|Customer_ID|feature_snapshot_date|Age|   Occupation|Annual_Income|Monthly_Inhand_Salary|Num_Bank_Accounts|Num_Credit_Card|Interest_Rate|Num_of_Loan|        Type_of_Loan|Delay_from_due_date|Num_of_Delayed_Payment|Changed_Credit_Limit|Num_Credit_Inquiries|Credit_Mix|Outstanding_Debt|Credit_Utilization_Ratio|  Credit_History_Age|Payment_of_Min_Amount|Total_EMI_per_month|Amount_invested_monthly|   Payment_Behaviour|Mo

In [8]:
# extract label store
features_sdf = features_store_sdf.filter((col("snapshot_date") >= config["train_test_start_date"]) & (col("snapshot_date") <= config["oot_end_date"]))

print("extracted features_sdf", features_sdf.count(), config["train_test_start_date"], config["oot_end_date"])

extracted features_sdf 5143 2023-07-01 00:00:00 2024-08-31 00:00:00


## prepare data for modeling

In [9]:
# prepare data for modeling
data_pdf = labels_sdf.join(features_sdf, on=["Customer_ID"], how="inner").toPandas()

data_pdf

,Customer_ID,label,label_def,snapshot_date,feature_snapshot_date,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,fe_11,fe_12,fe_13,fe_14,fe_15,fe_16,fe_17,fe_18,fe_19,fe_20
0,CUS_0x100b,0,30dpd_6mob,2024-09-01,2024-03-01,19,Media Manager,113781.390625,9549.782227,1,...,125,2,60,242,177,29,0,0,380,0
1,CUS_0x1096,1,30dpd_6mob,2024-09-01,2024-03-01,43,Lawyer,12931.559570,1348.630005,4,...,140,0,84,62,165,0,149,188,28,55
2,CUS_0x112d,1,30dpd_6mob,2024-09-01,2024-03-01,22,Accountant,14728.209961,1480.350830,10,...,0,55,171,290,85,0,0,122,165,0
3,CUS_0x12af,0,30dpd_6mob,2024-09-01,2024-03-01,37,Manager,33799.359375,2944.613281,5,...,0,90,292,197,80,131,23,102,179,153
4,CUS_0x12cb,1,30dpd_6mob,2024-09-01,2024-03-01,39,Writer,45188.519531,3868.709961,9,...,27,252,143,120,276,210,55,29,89,96
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5138,CUS_0xd91,0,30dpd_6mob,2024-03-01,2023-09-01,38,Musician,93939.359375,8023.279785,5,...,206,168,3,138,40,64,165,158,191,0
5139,CUS_0xdf6,0,30dpd_6mob,2024-03-01,2023-09-01,55,Mechanic,40453.851562,2772.811035,2,...,179,299,0,142,0,178,57,70,17,36
5140,CUS_0xe23,0,30dpd_6mob,2024-03-01,2023-09-01,39,Musician,17815.585938,1427.632080,8,...,77,124,190,0,282,25,53,85,80,50
5141,CUS_0xe4e,0,30dpd_6mob,2024-03-01,2023-09-01,22,Scientist,108073.679688,8738.139648,3,...,94,83,80,175,100,327,86,24,124,0


In [10]:
# split data into train - test - oot
oot_pdf = data_pdf[(data_pdf['snapshot_date'] >= config["oot_start_date"].date()) & (data_pdf['snapshot_date'] <= config["oot_end_date"].date())]
train_test_pdf = data_pdf[(data_pdf['snapshot_date'] >= config["train_test_start_date"].date()) & (data_pdf['snapshot_date'] <= config["train_test_end_date"].date())]

exclude_cols = [
    "Customer_ID", "label_def", "snapshot_date", "feature_snapshot_date", "label"
]
feature_cols = [c for c in data_pdf.columns if c not in exclude_cols]

print(f"Using {len(feature_cols)} feature columns")

X_oot = oot_pdf[feature_cols]
y_oot = oot_pdf["label"]
X_train, X_test, y_train, y_test = train_test_split(
    train_test_pdf[feature_cols], train_test_pdf["label"], 
    test_size= 1 - config["train_test_ratio"],
    random_state=88,     # Ensures reproducibility
    shuffle=True,        # Shuffle the data before splitting
    stratify=train_test_pdf["label"]           # Stratify based on the label column
)


print('X_train', X_train.shape[0])
print('X_test', X_test.shape[0])
print('X_oot', X_oot.shape[0])
print('y_train', y_train.shape[0], round(y_train.mean(),2))
print('y_test', y_test.shape[0], round(y_test.mean(),2))
print('y_oot', y_oot.shape[0], round(y_oot.mean(),2))

X_train

Using 43 feature columns
X_train 2018
X_test 505
X_oot 878
y_train 2018 0.28
y_test 505 0.28
y_oot 878 0.28


,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,...,fe_11,fe_12,fe_13,fe_14,fe_15,fe_16,fe_17,fe_18,fe_19,fe_20
2543,18,Musician,8814.919922,1011.576660,10,7,23,8,"Auto Loan, Payday Loan, Payday Loan, Debt Cons...",16,...,0,68,0,210,0,74,195,0,0,130
3931,55,Journalist,18524.519531,1739.709961,2,3,8,0,None,14,...,176,0,213,0,76,169,85,242,76,0
1834,32,Accountant,166907.718750,13788.976562,5,5,11,4,"Auto Loan, Mortgage Loan, Personal Loan, and S...",10,...,87,163,162,0,60,67,120,139,107,0
4726,28,Engineer,65455.160156,5194.596680,9,8,32,2,"Personal Loan, and Student Loan",20,...,134,190,101,12,0,82,6,159,278,463
4805,37,None,62683.589844,5047.632324,8,8,32,9,"Not Specified, Student Loan, Student Loan, Deb...",23,...,0,149,352,128,208,91,57,0,108,98
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1686,54,Media Manager,33991.621094,2973.635010,3,6,6,4,"Auto Loan, Auto Loan, Home Equity Loan, and De...",2,...,34,123,66,81,0,0,0,91,385,178
4277,52,Mechanic,33346.921875,2696.002441,5,3,1,0,None,6,...,125,41,220,198,12,99,36,90,118,58
4312,39,Accountant,68824.421875,4924.658691,4,1,4,1,Not Specified,11,...,191,50,114,82,0,184,389,79,97,323
4547,20,Developer,36349.281250,2885.106689,6,8,26,4,"Auto Loan, Not Specified, Credit-Builder Loan,...",21,...,0,19,274,205,42,65,54,0,116,286


## preprocess data

In [11]:
# Copy training, test, oot data
X_train_prep = X_train.copy()
X_test_prep = X_test.copy()
X_oot_prep = X_oot.copy()

# Encode categorical columns
cat_cols = X_train_prep.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns:", cat_cols)

for col in cat_cols:
    # Use category codes for simplicity
    all_values = pd.concat([X_train_prep[col], X_test_prep[col], X_oot_prep[col]], axis=0)
    mapping = {cat: i for i, cat in enumerate(all_values.astype('category').cat.categories)}
    X_train_prep[col] = X_train_prep[col].map(mapping)
    X_test_prep[col] = X_test_prep[col].map(mapping)
    X_oot_prep[col] = X_oot_prep[col].map(mapping)

# Replace NaN / inf
for df in [X_train_prep, X_test_prep, X_oot_prep]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(df.mean(), inplace=True)

# Fit scaler on training data ONLY
transformer_stdscaler = StandardScaler()
transformer_stdscaler.fit(X_train_prep)

X_train_processed = transformer_stdscaler.transform(X_train_prep)
X_test_processed = transformer_stdscaler.transform(X_test_prep)
X_oot_processed = transformer_stdscaler.transform(X_oot_prep)

# Quick validation
print('X_train_processed', X_train_processed.shape[0])
print('X_test_processed', X_test_processed.shape[0])
print('X_oot_processed', X_oot_processed.shape[0])

pd.DataFrame(X_train_processed, columns=X_train_prep.columns).head()


Categorical columns: ['Occupation', 'Type_of_Loan', 'Credit_Mix', 'Credit_History_Age', 'Payment_of_Min_Amount', 'Payment_Behaviour']
X_train_processed 2018
X_test_processed 505
X_oot_processed 878


,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,...,fe_11,fe_12,fe_13,fe_14,fe_15,fe_16,fe_17,fe_18,fe_19,fe_20
0,-1.625639,1.000623e+00,-0.122096,-0.996102,1.505711,-0.136190,-0.115223,-0.041384,-1.570373e+00,-0.307556,...,-1.260249,-0.489841,-1.251215,1.224109,-1.272175,-0.412266,0.982464,-1.275808,-1.211255,0.220387
1,1.976995,-2.146039e-01,-0.116078,-0.774621,-1.158493,-0.163823,-0.145291,-0.146200,2.122020e-16,-0.444307,...,0.763369,-1.290587,1.128739,-1.224988,-0.387543,0.674492,-0.287368,1.448223,-0.359797,-1.202994
2,-0.262480,-1.672876e+00,-0.024115,2.890492,-0.159417,-0.150006,-0.139277,-0.093792,-1.656234e+00,-0.717811,...,-0.259938,0.628850,0.558891,-1.224988,-0.573781,-0.492343,0.116669,0.288821,-0.012491,-1.202994
3,-0.651954,-7.006946e-01,-0.086992,0.276277,1.172686,-0.129281,-0.097182,-0.119996,1.444103e+00,-0.034052,...,0.280460,0.946793,-0.122692,-1.085039,-1.272175,-0.320750,-1.199339,0.513948,1.903289,3.866431
4,0.224362,-2.158676e-16,-0.088710,0.231574,0.839660,-0.129281,-0.097182,-0.028282,5.892234e-01,0.171075,...,-1.260249,0.463990,2.681855,0.267795,1.148924,-0.217794,-0.610599,-1.275808,-0.001288,-0.129984


## train model

In [12]:
# Define the XGBoost classifier
xgb_clf = xgb.XGBClassifier(eval_metric='logloss', random_state=88)

# Define the hyperparameter space to search
param_dist = {
    'n_estimators': [25, 50],
    'max_depth': [2, 3],  # lower max_depth to simplify the model
    'learning_rate': [0.01, 0.1],
    'subsample': [0.6, 0.8],
    'colsample_bytree': [0.6, 0.8],
    'gamma': [0, 0.1],
    'min_child_weight': [1, 3, 5],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

# Create a scorer based on AUC score
auc_scorer = make_scorer(roc_auc_score)

# Set up the random search with cross-validation
random_search = RandomizedSearchCV(
    estimator=xgb_clf,
    param_distributions=param_dist,
    scoring=auc_scorer,
    n_iter=100,  # Number of iterations for random search
    cv=3,       # Number of folds in cross-validation
    verbose=1,
    random_state=42,
    n_jobs=-1   # Use all available cores
)

# Perform the random search
random_search.fit(X_train_processed, y_train)

# Output the best parameters and best score
print("Best parameters found: ", random_search.best_params_)
print("Best AUC score: ", random_search.best_score_)

# Evaluate the model on the train set
best_model = random_search.best_estimator_
y_pred_proba = best_model.predict_proba(X_train_processed)[:, 1]
train_auc_score = roc_auc_score(y_train, y_pred_proba)
print("Train AUC score: ", train_auc_score)

# Evaluate the model on the test set
best_model = random_search.best_estimator_
y_pred_proba = best_model.predict_proba(X_test_processed)[:, 1]
test_auc_score = roc_auc_score(y_test, y_pred_proba)
print("Test AUC score: ", test_auc_score)

# Evaluate the model on the oot set
best_model = random_search.best_estimator_
y_pred_proba = best_model.predict_proba(X_oot_processed)[:, 1]
oot_auc_score = roc_auc_score(y_oot, y_pred_proba)
print("OOT AUC score: ", oot_auc_score)

print("TRAIN GINI score: ", round(2*train_auc_score-1,3))
print("Test GINI score: ", round(2*test_auc_score-1,3))
print("OOT GINI score: ", round(2*oot_auc_score-1,3))

Fitting 3 folds for each of 100 candidates, totalling 300 fits
Best parameters found:  {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 50, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 0.8}
Best AUC score:  0.693497479802956
Train AUC score:  0.9124923106907283
Test AUC score:  0.7846195631086796
OOT AUC score:  0.8026369426751592
TRAIN GINI score:  0.825
Test GINI score:  0.569
OOT GINI score:  0.605


## prepare model artefact to save

In [13]:
model_artefact = {}

model_artefact['model'] = best_model
model_artefact['model_version'] = "credit_model_"+config["model_train_date_str"].replace('-','_')
model_artefact['preprocessing_transformers'] = {}
model_artefact['preprocessing_transformers']['stdscaler'] = transformer_stdscaler
model_artefact['data_dates'] = config
model_artefact['data_stats'] = {}
model_artefact['data_stats']['X_train'] = X_train.shape[0]
model_artefact['data_stats']['X_test'] = X_test.shape[0]
model_artefact['data_stats']['X_oot'] = X_oot.shape[0]
model_artefact['data_stats']['y_train'] = round(y_train.mean(),2)
model_artefact['data_stats']['y_test'] = round(y_test.mean(),2)
model_artefact['data_stats']['y_oot'] = round(y_oot.mean(),2)
model_artefact['results'] = {}
model_artefact['results']['auc_train'] = train_auc_score
model_artefact['results']['auc_test'] = test_auc_score
model_artefact['results']['auc_oot'] = oot_auc_score
model_artefact['results']['gini_train'] = round(2*train_auc_score-1,3)
model_artefact['results']['gini_test'] = round(2*test_auc_score-1,3)
model_artefact['results']['gini_oot'] = round(2*oot_auc_score-1,3)
model_artefact['hp_params'] = random_search.best_params_


pprint.pprint(model_artefact)

{'data_dates': {'model_train_date': datetime.datetime(2024, 9, 1, 0, 0),
                'model_train_date_str': '2024-09-01',
                'oot_end_date': datetime.datetime(2024, 8, 31, 0, 0),
                'oot_period_months': 2,
                'oot_start_date': datetime.datetime(2024, 7, 1, 0, 0),
                'train_test_end_date': datetime.datetime(2024, 6, 30, 0, 0),
                'train_test_period_months': 12,
                'train_test_ratio': 0.8,
                'train_test_start_date': datetime.datetime(2023, 7, 1, 0, 0)},
 'data_stats': {'X_oot': 878,
                'X_test': 505,
                'X_train': 2018,
                'y_oot': np.float64(0.28),
                'y_test': np.float64(0.28),
                'y_train': np.float64(0.28)},
 'hp_params': {'colsample_bytree': 0.8,
               'gamma': 0.1,
               'learning_rate': 0.1,
               'max_depth': 3,
               'min_child_weight': 3,
               'n_estimators': 50,
          

## save artefact to model bank

In [14]:
# create model_bank dir
model_bank_directory = "model_bank/auto_ml/"

if not os.path.exists(model_bank_directory):
    os.makedirs(model_bank_directory)

In [15]:
# Full path to the file
file_path = os.path.join(model_bank_directory, model_artefact['model_version'] + '.pkl')

# Write the model to a pickle file
with open(file_path, 'wb') as file:
    pickle.dump(model_artefact, file)

print(f"Model saved to {file_path}")


Model saved to model_bank/credit_model_2024_09_01.pkl


## test load pickle and make model inference

In [16]:
# Load the model from the pickle file
with open(file_path, 'rb') as file:
    loaded_model_artefact = pickle.load(file)

y_pred_proba = loaded_model_artefact['model'].predict_proba(X_oot_processed)[:, 1]
oot_auc_score = roc_auc_score(y_oot, y_pred_proba)
print("OOT AUC score: ", oot_auc_score)

print("Model loaded successfully!")

OOT AUC score:  0.8026369426751592
Model loaded successfully!
